In [ ]:
{
 "cells": [
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "sys.path.append('/opt/workspace')\n",
    "\n",
    "from datetime import datetime, timedelta\n",
    "from connectors.clickhouse_client import ClickHouseClient"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "ref_date = datetime.now().strftime(\"%Y-%m-%d\")\n",
    "previous_date = (datetime.now() - timedelta(days=1)).strftime(\"%Y-%m-%d\")\n",
    "\n",
    "ref_date, previous_date"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "client = ClickHouseClient()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "table_name = \"YOUR_TABLE_NAME\""
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "inserts_query = f\"\"\"\n",
    "INSERT INTO silver.delta_events (\n",
    "    ref_date, table_name, primary_key, operation_type, \n",
    "    row_hash_after, data_after\n",
    ")\n",
    "SELECT\n",
    "    toDate('{ref_date}') AS ref_date,\n",
    "    '{table_name}' AS table_name,\n",
    "    current.primary_key,\n",
    "    'INSERT' AS operation_type,\n",
    "    current.row_hash AS row_hash_after,\n",
    "    current.data AS data_after\n",
    "FROM bronze.snapshot_raw AS current\n",
    "LEFT JOIN bronze.snapshot_raw AS previous\n",
    "    ON current.primary_key = previous.primary_key\n",
    "    AND current.table_name = previous.table_name\n",
    "    AND previous.ref_date = toDate('{previous_date}')\n",
    "WHERE current.ref_date = toDate('{ref_date}')\n",
    "    AND current.table_name = '{table_name}'\n",
    "    AND previous.primary_key IS NULL\n",
    "\"\"\"\n",
    "\n",
    "client.execute_query(inserts_query)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "inserts_count = client.execute_query_with_result(\n",
    "    f\"\"\"\n",
    "    SELECT count() as total\n",
    "    FROM silver.delta_events\n",
    "    WHERE ref_date = toDate('{ref_date}')\n",
    "        AND table_name = '{table_name}'\n",
    "        AND operation_type = 'INSERT'\n",
    "    \"\"\"\n",
    ")\n",
    "inserts_count.result_rows"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "updates_query = f\"\"\"\n",
    "INSERT INTO silver.delta_events (\n",
    "    ref_date, table_name, primary_key, operation_type,\n",
    "    row_hash_before, row_hash_after, data_before, data_after\n",
    ")\n",
    "SELECT\n",
    "    toDate('{ref_date}') AS ref_date,\n",
    "    '{table_name}' AS table_name,\n",
    "    current.primary_key,\n",
    "    'UPDATE' AS operation_type,\n",
    "    previous.row_hash AS row_hash_before,\n",
    "    current.row_hash AS row_hash_after,\n",
    "    previous.data AS data_before,\n",
    "    current.data AS data_after\n",
    "FROM bronze.snapshot_raw AS current\n",
    "INNER JOIN bronze.snapshot_raw AS previous\n",
    "    ON current.primary_key = previous.primary_key\n",
    "    AND current.table_name = previous.table_name\n",
    "    AND previous.ref_date = toDate('{previous_date}')\n",
    "WHERE current.ref_date = toDate('{ref_date}')\n",
    "    AND current.table_name = '{table_name}'\n",
    "    AND current.row_hash != previous.row_hash\n",
    "\"\"\"\n",
    "\n",
    "client.execute_query(updates_query)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "updates_count = client.execute_query_with_result(\n",
    "    f\"\"\"\n",
    "    SELECT count() as total\n",
    "    FROM silver.delta_events\n",
    "    WHERE ref_date = toDate('{ref_date}')\n",
    "        AND table_name = '{table_name}'\n",
    "        AND operation_type = 'UPDATE'\n",
    "    \"\"\"\n",
    ")\n",
    "updates_count.result_rows"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "deletes_query = f\"\"\"\n",
    "INSERT INTO silver.delta_events (\n",
    "    ref_date, table_name, primary_key, operation_type,\n",
    "    row_hash_before, data_before\n",
    ")\n",
    "SELECT\n",
    "    toDate('{ref_date}') AS ref_date,\n",
    "    '{table_name}' AS table_name,\n",
    "    previous.primary_key,\n",
    "    'DELETE' AS operation_type,\n",
    "    previous.row_hash AS row_hash_before,\n",
    "    previous.data AS data_before\n",
    "FROM bronze.snapshot_raw AS previous\n",
    "LEFT JOIN bronze.snapshot_raw AS current\n",
    "    ON previous.primary_key = current.primary_key\n",
    "    AND previous.table_name = current.table_name\n",
    "    AND current.ref_date = toDate('{ref_date}')\n",
    "WHERE previous.ref_date = toDate('{previous_date}')\n",
    "    AND previous.table_name = '{table_name}'\n",
    "    AND current.primary_key IS NULL\n",
    "\"\"\"\n",
    "\n",
    "client.execute_query(deletes_query)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "deletes_count = client.execute_query_with_result(\n",
    "    f\"\"\"\n",
    "    SELECT count() as total\n",
    "    FROM silver.delta_events\n",
    "    WHERE ref_date = toDate('{ref_date}')\n",
    "        AND table_name = '{table_name}'\n",
    "        AND operation_type = 'DELETE'\n",
    "    \"\"\"\n",
    ")\n",
    "deletes_count.result_rows"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "update_state_query = f\"\"\"\n",
    "INSERT INTO silver.current_state (\n",
    "    table_name, primary_key, row_hash, data,\n",
    "    first_seen_date, last_seen_date, is_active\n",
    ")\n",
    "SELECT\n",
    "    table_name,\n",
    "    primary_key,\n",
    "    row_hash,\n",
    "    data,\n",
    "    ref_date AS first_seen_date,\n",
    "    ref_date AS last_seen_date,\n",
    "    1 AS is_active\n",
    "FROM bronze.snapshot_raw\n",
    "WHERE ref_date = toDate('{ref_date}')\n",
    "    AND table_name = '{table_name}'\n",
    "\"\"\"\n",
    "\n",
    "client.execute_query(update_state_query)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "summary = client.execute_query_with_result(\n",
    "    f\"\"\"\n",
    "    SELECT\n",
    "        operation_type,\n",
    "        count() as total\n",
    "    FROM silver.delta_events\n",
    "    WHERE ref_date = toDate('{ref_date}')\n",
    "        AND table_name = '{table_name}'\n",
    "    GROUP BY operation_type\n",
    "    \"\"\"\n",
    ")\n",
    "summary.result_rows"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "client.close()"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.11.14"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 2
}